# STR-2 — Idempotency, checkpoints & restart

**Break → Detect → Fix → Prove.** A streaming job must survive restarts **without losing or
double-processing data**. Structured Streaming's **checkpoint** stores the Kafka offsets it has
committed (plus any state); on restart the query resumes from exactly those offsets. Pair that with
an **idempotent sink** (Iceberg) and you get effectively **exactly-once** ingestion across crashes,
redeploys, and re-runs.

We demonstrate this the way it behaves in production: a sequence of **bounded `availableNow` runs
that all share ONE checkpoint** — each run is a "restart".

**Pre-requisite:** `make up` (Kafka at `localhost:29092` for producers, `kafka:9092` for Spark).
Inspect the topic live in **kafka-ui** at http://localhost:8080; the Spark UI is at
http://localhost:4040. **Laptop-safe:** bounded produces (1000 + 500) and `trigger(availableNow=True)`
reads (drain-and-stop, never unbounded). Checkpoint + table under `.tmp/`; Teardown deletes both.

See the companion [`README.md`](./README.md) and the [troubleshooting sheet](../../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.kafka_helpers import ensure_topic, produce_events, topic_end_offsets, delete_topic, SPARK_BOOTSTRAP
from common.iceberg_meta import table_health
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, DoubleType

TOPIC    = "str2_events"
SINK     = "iceberg_catalog.default.str2_sink"        # idempotent target (namespace.table)
SCRATCH  = "iceberg_catalog.default.str2_sink_bad"    # contrast: what a wiped checkpoint does
CKPT     = ".tmp/checkpoint_str2"                     # Spark-side path (server CWD=/app -> repo .tmp)
CKPT_BAD = ".tmp/checkpoint_str2_BAD"                 # a NEW/empty checkpoint for the contrast

print("Spark reads via", SPARK_BOOTSTRAP, "| kafka-ui:", "http://localhost:8080", "| Spark UI:", "http://localhost:4040")

## Step 0 — a fresh topic, no sink, one checkpoint

Start clean so the run-by-run counts are deterministic: recreate the topic and drop both tables.
The event value is JSON `{"id": i, "v": i*1.0}` — `id` is our dedup key. `stream_query` builds the
**same** bounded stream each time (Kafka → parse JSON → append to Iceberg); `trigger(availableNow=True)`
drains all available data then **stops on its own**. Re-running this notebook? `make clean` first to
clear any stale `checkpoint_str2` dir (dropping a table does **not** remove its checkpoint).

In [ ]:
ensure_topic(TOPIC, num_partitions=1, recreate=True)
spark.sql(f"DROP TABLE IF EXISTS {SINK}")
spark.sql(f"DROP TABLE IF EXISTS {SCRATCH}")
print("Using checkpoint:", CKPT, "(run `make clean` first if a previous STR-2 run left state)")

schema = StructType([StructField("id", LongType(), True), StructField("v", DoubleType(), True)])

def stream_query(checkpoint, table):
    """The SAME bounded stream every run: Kafka -> parse JSON -> append to Iceberg.
    startingOffsets is only consulted on a FRESH checkpoint; availableNow = drain-and-stop."""
    raw = (spark.readStream.format("kafka")
           .option("kafka.bootstrap.servers", SPARK_BOOTSTRAP)
           .option("subscribe", TOPIC)
           .option("startingOffsets", "earliest")
           .load())
    parsed = raw.select(F.from_json(F.col("value").cast("string"), schema).alias("d")).select("d.*")
    q = (parsed.writeStream.format("iceberg")
         .option("checkpointLocation", checkpoint)
         .trigger(availableNow=True)
         .toTable(table))
    q.awaitTermination()      # bounded: returns once the available data is drained
    return q

def counts(table):
    r = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT id) AS d FROM {table}").first()
    return int(r["n"]), int(r["d"])

def offsets(q, label):
    lp = q.lastProgress or {}
    src = (lp.get("sources") or [{}])[0]
    print(f"{label}: numInputRows={lp.get('numInputRows')}  start={src.get('startOffset')}  end={src.get('endOffset')}")

## 1. First start — produce 1000, run the stream

Produce a bounded batch of 1000 events, then run the `availableNow` stream once. With no prior
checkpoint it starts from `earliest`, drains all 1000, **commits the offsets**, and stops. The sink
should hold exactly 1000 rows. `offsets()` (from `q.lastProgress`) shows the range this run read —
the first run starts at offset 0.

In [ ]:
produce_events(TOPIC, 1000)
print("topic end offsets after produce #1:", topic_end_offsets(TOPIC))

q1 = stream_query(CKPT, SINK)
offsets(q1, "run 1")
n, d = counts(SINK)
print(f"RUN 1  -> sink COUNT(*)={n}  COUNT(DISTINCT id)={d}   (expected 1000 / 1000)")

## 2. Restart safety — produce 500 MORE, run the SAME stream (SAME checkpoint)

The realistic case: more data arrived and the job restarts. Produce **500 more** events and run the
**same** query against the **same** `checkpointLocation`. It resumes from the committed offsets and
ingests **only the new 500** — the resuming run's **start offset == run 1's end offset**, and
`numInputRows` ≈ 500. No re-read of the first 1000, no duplicates.

In [ ]:
produce_events(TOPIC, 500, value_fn=lambda i: {"id": 1000 + i, "v": (1000 + i) * 1.0})
print("topic end offsets after produce #2:", topic_end_offsets(TOPIC))

q2 = stream_query(CKPT, SINK)   # SAME checkpoint -> resumes from committed offsets
offsets(q2, "run 2")            # start should equal run 1's end
n, d = counts(SINK)
print(f"RUN 2  -> sink COUNT(*)={n}  COUNT(DISTINCT id)={d}   (expected 1500 / 1500, no dupes)")

## 3. Idempotent restart — run a THIRD time with NO new data

A bare restart with nothing new in the topic. The query resumes from the committed offsets, finds
nothing past them, ingests **0 rows** (`numInputRows == 0`, start == end), and the count stays at
1500. Restarting reprocesses nothing.

In [ ]:
q3 = stream_query(CKPT, SINK)   # SAME checkpoint, no new data produced
offsets(q3, "run 3")            # numInputRows == 0
n, d = counts(SINK)
print(f"RUN 3  -> sink COUNT(*)={n}  COUNT(DISTINCT id)={d}   (expected 1500 / 1500, unchanged)")

## 4. Break it / contrast — a NEW/empty checkpoint reprocesses everything

The mistake people make under pressure: *"the stream is stuck — let me wipe the checkpoint and
restart."* Point the **same** stream at a **fresh** `checkpointLocation` (writing to a scratch table
so `SINK` stays intact). With no committed offsets and `startingOffsets=earliest`, it re-reads the
topic from offset 0 and **reprocesses all 1500** — duplicates. The checkpoint is exactly what made
restart safe; deleting it throws that away.

In [ ]:
q_bad = stream_query(CKPT_BAD, SCRATCH)   # NEW/empty checkpoint -> earliest -> re-reads all
offsets(q_bad, "contrast")                # numInputRows == 1500, start == 0
n, d = counts(SCRATCH)
print(f"CONTRAST (wiped checkpoint) -> scratch COUNT(*)={n}  COUNT(DISTINCT id)={d}")
print("=> reprocessed from offset 0: every event read again (a real keyed/append sink would now have dupes).")

## 5. Prove it

Exactly-once into Iceberg: counts go **1000 → 1500 → 1500** with `COUNT(*) == COUNT(DISTINCT id)` at
every step (no duplicate `id`s). Run 2 ingested only the new 500 (resumed from committed offsets);
run 3 reprocessed nothing (idempotent restart). The contrast shows what a wiped checkpoint does.

In [ ]:
sink_n, sink_d = counts(SINK)
print("SINK (one durable checkpoint, three restarts):")
print(f"   COUNT(*)           = {sink_n}")
print(f"   COUNT(DISTINCT id) = {sink_d}")
print(f"   duplicates         = {sink_n - sink_d}   <- 0 == exactly-once")
print(f"   sequence observed  : 1000 -> 1500 -> 1500")

h = table_health(spark, SINK, "STR-2 sink (after 3 restarts)")
print("\ntable_health:", {k: h[k] for k in ("data_files", "snapshots", "manifests")})
print("\nverdict:", "EXACTLY-ONCE (checkpoint + idempotent sink)" if sink_n == sink_d == 1500
      else "unexpected — re-run from Step 0 after `make clean`")

## Takeaways & "in real production…"

- **Never delete a checkpoint to "reset" a stream casually** — with `earliest` it reprocesses
  (duplicates), with `latest` it would skip everything already in the topic. A wipe is a deliberate,
  last-resort reset, paired with a sink that tolerates the replay.
- **Checkpoint + idempotent sink = exactly-once into Iceberg.** The checkpoint guarantees
  at-least-once delivery of each offset range; the atomic/idempotent sink removes duplicates. Keying
  the write (`MERGE` by `id`) makes even a full replay safe — that's the CDC-7 pattern.
- **One stable `checkpointLocation` per query**, on durable storage, owned alongside the job's
  state. Changing the query shape can invalidate it — version your streaming jobs.
- **Ties to KAF-2** (the checkpoint *is* the offset store for Structured Streaming) and to **CAP**
  (restart/replay drills in the incident simulator).

## Teardown

We created a topic, an Iceberg sink, and a scratch contrast table — delete them. `make clean`
removes everything under `.tmp/`, including the `checkpoint_str2` directories (dropping a table does
**not** remove its checkpoint, so clear them before re-running this notebook).

In [ ]:
delete_topic(TOPIC)
spark.sql(f"DROP TABLE IF EXISTS {SINK}")
spark.sql(f"DROP TABLE IF EXISTS {SCRATCH}")
print("Deleted topic", TOPIC, "and dropped", SINK, "/", SCRATCH)
print("Run `make clean` to clear the checkpoints under .tmp/ (checkpoint_str2 / checkpoint_str2_BAD).")